In [1]:
# CELL 1 — Imports & config

import requests
import json
import csv
import os
import time
from tqdm import tqdm

DATA_DIR = "../data"
EPSS_FILE = f"{DATA_DIR}/epss_scores.csv"
EPSS_API = "https://api.first.org/data/v1/epss"
PAGE_SIZE = 10000

print(f"Output: {EPSS_FILE}")

Output: ../data/epss_scores.csv


In [2]:
# CELL 2 — Check API and get total CVE count

r = requests.get(EPSS_API, params={"limit": 1, "offset": 0}, timeout=30)
print(f"Status: {r.status_code}")

data = r.json()
total = data["total"]
print(f"Total CVEs with EPSS scores: {total:,}")
print(f"Score date: {data['data'][0]['date']}")
print(f"Pages needed at {PAGE_SIZE}/page: {-(-total // PAGE_SIZE)}")

Status: 200
Total CVEs with EPSS scores: 341,601
Score date: 2026-06-22
Pages needed at 10000/page: 35


In [3]:
# CELL 3 — Download all EPSS scores (paginated)
# Downloads the full EPSS database then filters to our CVEs in Cell 5.
# ~25 requests at 10k/page is much faster than ~5,200 batch requests for 156k CVE IDs.

epss_lookup = {}  # cve_id -> {"epss": float, "percentile": float, "date": str}
offset = 0

with tqdm(total=total, unit="CVE", desc="Downloading EPSS") as pbar:
    while offset < total:
        r = requests.get(
            EPSS_API,
            params={"limit": PAGE_SIZE, "offset": offset},
            timeout=60
        )

        if r.status_code != 200:
            print(f"\n❌ Error {r.status_code} at offset {offset}")
            break

        batch = r.json()["data"]
        if not batch:
            break

        for row in batch:
            epss_lookup[row["cve"]] = {
                "epss": float(row["epss"]),
                "percentile": float(row["percentile"]),
                "date": row["date"]
            }

        offset += len(batch)
        pbar.update(len(batch))
        time.sleep(0.2)

print(f"\n✅ Downloaded {len(epss_lookup):,} EPSS scores")
epss_date = next(iter(epss_lookup.values()))["date"]
print(f"Score date: {epss_date}")


✅ Downloaded 341,601 EPSS scores
Score date: 2026-06-22


In [4]:
# CELL 4 — Load all CVE IDs from our dataset

OUR_FILES = [
    f"{DATA_DIR}/cves_all_critical.jsonl",
    f"{DATA_DIR}/cves_all_high.jsonl",
    f"{DATA_DIR}/cves_all_medium.jsonl",
    f"{DATA_DIR}/cves_all_low.jsonl",
]

our_cve_ids = set()
for path in OUR_FILES:
    with open(path) as f:
        for line in f:
            our_cve_ids.add(json.loads(line)["id"])

print(f"Total unique CVE IDs in our dataset: {len(our_cve_ids):,}")
print(f"CVEs also in EPSS:     {sum(1 for c in our_cve_ids if c in epss_lookup):,}")
print(f"CVEs NOT in EPSS:      {sum(1 for c in our_cve_ids if c not in epss_lookup):,}")

Total unique CVE IDs in our dataset: 156,084
CVEs also in EPSS:     156,073
CVEs NOT in EPSS:      11


In [5]:
# CELL 5 — Filter to our CVEs and save as CSV

rows_written = 0
rows_missing = 0

with open(EPSS_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["cve_id", "epss", "percentile", "date"])
    for cve_id in sorted(our_cve_ids):
        if cve_id in epss_lookup:
            e = epss_lookup[cve_id]
            writer.writerow([cve_id, e["epss"], e["percentile"], e["date"]])
            rows_written += 1
        else:
            writer.writerow([cve_id, "", "", ""])
            rows_missing += 1

print(f"✅ Saved {EPSS_FILE}")
print(f"   Rows with EPSS score: {rows_written:,}")
print(f"   Rows without score:   {rows_missing:,} (CVE too recent or not yet scored)")

✅ Saved ../data/epss_scores.csv
   Rows with EPSS score: 156,073
   Rows without score:   11 (CVE too recent or not yet scored)


In [6]:
# CELL 6 — Summary stats

import statistics

scores = [epss_lookup[c]["epss"] for c in our_cve_ids if c in epss_lookup]

print(f"EPSS score distribution across our {len(scores):,} matched CVEs:")
print(f"  Min:    {min(scores):.4f}")
print(f"  Median: {statistics.median(scores):.4f}")
print(f"  Mean:   {statistics.mean(scores):.4f}")
print(f"  Max:    {max(scores):.4f}")
print()
bands = [
    ("EPSS >= 0.5  (very high)", 0.5, 1.0),
    ("EPSS 0.1-0.5 (elevated) ", 0.1, 0.5),
    ("EPSS < 0.1   (low)      ", 0.0, 0.1),
]
for label, lo, hi in bands:
    n = sum(1 for s in scores if lo <= s < hi)
    print(f"  {label}: {n:,} ({n/len(scores)*100:.1f}%)")

EPSS score distribution across our 156,073 matched CVEs:
  Min:    0.0005
  Median: 0.0049
  Mean:   0.0201
  Max:    1.0000

  EPSS >= 0.5  (very high): 1,651 (1.1%)
  EPSS 0.1-0.5 (elevated) : 2,857 (1.8%)
  EPSS < 0.1   (low)      : 151,565 (97.1%)
